# ForkliftGym quickstart
Install the project in your notebook kernel first. In a cloned folder use `%pip install -e ".[sb3]"`; in Colab or another folder use `%pip install "forklift-gym[sb3] @ git+https://github.com/luisgarrote/ForkliftGym.git"`.

This notebook inspects the real RGB view and 44-D observation, runs scripted traffic, then uses the reproducible PPO launcher.


In [ ]:
import gymnasium as gym
import forklift_gym
from PIL import Image

env = gym.make("Forklift-v0", render_mode="rgb_array")
try:
    obs, info = env.reset(seed=7)
    print("Observation:", obs.shape, "Action:", env.action_space)
    print("Lidar:", obs[:36])
    print("Goal/motion/task features:", obs[36:])
    display(Image.fromarray(env.render()))
finally:
    env.close()


## Other agents moving
PPO controls the forklift. The two circular robots below run scripted patrol policies on every simulation step. Zero forklift control lets us inspect their movement.


In [ ]:
import numpy as np
from forklift_gym import SB3ForkliftEnv

env = SB3ForkliftEnv(num_dynamic_agents=2, obstacle_count=2, num_boxes=4,
                     render_mode="rgb_array")
try:
    env.reset(seed=7)
    for _ in range(40):
        obs, reward, terminated, truncated, info = env.step(np.zeros(3, dtype=np.float32))
        if terminated or truncated:
            break
    display(Image.fromarray(env.render()))
finally:
    env.close()


## Train a PPO policy
The launcher saves policy weights, normalization, scenario settings and raw Monitor logs. The short budget demonstrates the pipeline; it does not guarantee successful pallet delivery. Use a new output folder for each run.


In [ ]:
import subprocess
import sys
from pathlib import Path

run = Path("outputs/notebook_seed0")
subprocess.run([sys.executable, "-m", "forklift_gym", "train",
                "--steps", "10000", "--envs", "2",
                "--output", str(run)], check=True)


## Evaluate on held-out seeds
Evaluation restores the same history and normalization, freezes statistics and saves every episode to JSON. Fixed scenes can reset to the same initial geometry across seeds; generated scenes are needed for a generalization comparison.


In [ ]:
import json
subprocess.run([sys.executable, "-m", "forklift_gym", "evaluate",
                "--run", str(run), "--episodes", "5", "--seed", "10000"], check=True)
results = json.loads((run / "evaluation_seed10000_n5.json").read_text())
print("Success rate:", results["success_rate"], "Mean raw return:", results["mean_return"])
results["episodes"]


## Next experiments
Try `--traffic 2 --obstacles 2` in a separate training run, compare `--history 1` with `--history 4`, and repeat with three training seeds. `--vec-env subproc` enables simulator processes; benchmark a small number of workers first. Keep the total transition budget consistent across conditions.
